<a href="https://colab.research.google.com/github/RusieckiFilip/Inzynieria_Obliczeniowa_UW/blob/main/ATC_BADA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json

with open('all_flights_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)   # tu dostaniesz dokładny błąd z numerem linii/kolumny


In [ ]:
import json
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Funkcja obliczająca odległość w kilometrach między dwoma punktami geograficznymi
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Promień Ziemi w km
    phi1, phi2 = radians(lat1), radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)
    a = sin(delta_phi / 2)**2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Wczytanie danych
with open('all_flights_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Filtrowanie tylko pozycji + usunięcie wierszy z brakami
df = df[df['type'] == 'position']
df_clean = df.dropna(subset=['lat', 'lon', 'alt', 'gs', 'vertRate', 'aircrafttype','clock', 'id', 'orig', 'dest', 'ident']).copy()

# Konwersje typów
df_clean['lat'] = df_clean['lat'].astype(float)
df_clean['lon'] = df_clean['lon'].astype(float)
df_clean['alt'] = df_clean['alt'].astype(float)
df_clean['gs'] = df_clean['gs'].astype(float)
df_clean['clock'] = df_clean['clock'].astype(int)

# Sortowanie danych
df_sorted = df_clean.sort_values(['ident', 'clock'])
grouped = df_sorted.groupby('ident')

# Segmentacja
for ident, group in grouped:
    group = group.reset_index(drop=True)
    segments = []
    current_segment = [group.loc[0]]

    for i in range(1, len(group)):
        lat1, lon1 = group.loc[i - 1, ['lat', 'lon']]
        lat2, lon2 = group.loc[i, ['lat', 'lon']]
        alt1 = group.loc[i - 1, 'alt']
        time1 = group.loc[i - 1, 'clock']
        time2 = group.loc[i, 'clock']
        distance = haversine(lat1, lon1, lat2, lon2)
        time_diff = time2 - time1

        # Ustalanie progów zależnych od wysokości
        if alt1 > 30000:
            distance_threshold = 100  # km
            time_threshold = 600     # sekundy
        elif alt1 > 20000:
            distance_threshold = 70
            time_threshold = 500
        elif alt1 > 10000:
            distance_threshold = 50
            time_threshold = 400
        else:
            distance_threshold = 40
            time_threshold = 300

        # Sprawdzenie, czy rozpocząć nowy segment
        if distance > distance_threshold or time_diff > time_threshold:
            segments.append(pd.DataFrame(current_segment))
            current_segment = [group.loc[i]]
        else:
            current_segment.append(group.loc[i])

    # Dodaj ostatni segment
    if current_segment:
        segments.append(pd.DataFrame(current_segment))

    # Wyświetl
    for idx, segment in enumerate(segments, start=1):
        print(f"\n{'-'*50}\n")
        print(f"Przelot: {ident} | Segment: {idx}")
        print(segment[['clock', 'aircrafttype', 'ident',  'orig', 'dest', 'lat', 'lon', 'alt', 'gs', 'vertRate' ]])


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
2   502.0        0  
3   493.0       64  
4   496.0        0  
5   490.0        0  
6   490.0        0  
7   492.0       64  
8   496.0        0  
9   498.0        0  
10  498.0        0  
11  490.0       64  
12  483.0        0  
13  495.0        0  
14  476.0      -64  
15  475.0      -64  
16  474.0        0  
17  474.0        0  

--------------------------------------------------

Przelot: WZZ1175 | Segment: 2
         clock aircrafttype    ident  orig  dest       lat      lon      alt  \
18  1726575821         A321  WZZ1175  EPKT  LEBL  41.07844  9.39331  34975.0   
19  1726575911         A321  WZZ1175  EPKT  LEBL  41.03746  9.16663  34950.0   
20  1726575941         A321  WZZ1175  EPKT  LEBL  41.02377  9.09192  34975.0   
21  1726576031         A321  WZZ1175  EPKT  LEBL  40.98226  8.86428  35000.0   
22  1726576062         A321  WZZ1175  EPKT  LEBL  40.96811  8.78394  34975.0   
23  1726576123         A321  WZZ1175

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
import folium
from folium import Popup, DivIcon

# ------------------------------
# 1. Funkcja haversine (odległość km)
# ------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # km
    phi1, phi2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(lon2 - lon1)
    a = sin(dphi/2)**2 + cos(phi1)*cos(phi2)*sin(dlambda/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

# 2. Wczytanie i konwersje
with open('all_flights_2.json', 'r', encoding='utf-8') as f:
    data=json.load(f)
df=pd.DataFrame(data)
df=df[df['type']=='position'].dropna(subset=['lat','lon','alt','gs','vertRate','clock','ident']).copy()

# <-- TU DODANE RZUTOWANIA NA LICZBY -->
df[['lat','lon','alt','gs','vertRate']] = df[['lat','lon','alt','gs','vertRate']].apply(pd.to_numeric, errors='coerce')
df['clock'] = pd.to_datetime(df['clock'].astype(int), unit='s', errors='coerce')
df = df.dropna(subset=['lat','lon','alt','gs','vertRate','clock'])

df = df.sort_values(['ident','clock']).reset_index(drop=True)

# ------------------------------
# 3. Segmentacja per ident
# ------------------------------
all_segments = []  # lista tuple (ident, segment_idx, df_segment)
for ident, grp in df.groupby('ident'):
    grp = grp.reset_index(drop=True)
    segments = []
    current = [grp.loc[0]]
    for i in range(1, len(grp)):
        prev, curr = grp.loc[i-1], grp.loc[i]
        dist = haversine(prev.lat, prev.lon, curr.lat, curr.lon)
        dt = (curr.clock - prev.clock).total_seconds()
        # dobór progów wg wysokości poprzedniej próbki
        alt = prev.alt
        if   alt>30000: d_thr, t_thr = 100, 600
        elif alt>20000: d_thr, t_thr =  70, 500
        elif alt>10000: d_thr, t_thr =  50, 400
        else:            d_thr, t_thr =  40, 300
        if dist>d_thr or dt>t_thr:
            segments.append(pd.DataFrame(current))
            current = [curr]
        else:
            current.append(curr)
    if current:
        segments.append(pd.DataFrame(current))

    for idx, seg in enumerate(segments, 1):
        all_segments.append((ident, idx, seg))

# ------------------------------
# 4. Parametry prędkości per typ
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}

def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    grp['phase'] = np.nan

    params = aircraft_speeds.get(grp.at[0, 'aircrafttype'])
    if not params:
        return grp.fillna({'phase': 'Unknown'})

    FL = params["FL_CRUISE"]
    Vcl1 = params["Vcl1"]
    Vcr2 = params["Vcr2"]
    Vdes1 = params["Vdes1"]

    for i, row in grp.iterrows():
        alt = row['alt']
        gs = row['gs']
        roc = row['vertRate']

        # Climb
        if 100 <= alt <= FL - 4000 and roc >= 250:
            if gs <= Vcl1:
                grp.at[i, 'phase'] = "Lower Climb"
            else:
                grp.at[i, 'phase'] = "Upper Climb"

        # Descent
        elif 1000 <= alt <= FL - 5000 and roc <= -350:
            if gs <= Vdes1 or (alt < 10000 and gs <= 250):
                grp.at[i, 'phase'] = "Lower Descent"
            else:
                grp.at[i, 'phase'] = "Upper Descent"

        # Cruise
        else:
            if gs <= Vcr2:
                grp.at[i, 'phase'] = "Lower Cruise"
            else:
                grp.at[i, 'phase'] = "Upper Cruise"



    # Wypełnij ewentualne NaN-y z przodu/tyłu serii:
    grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna('Unknown')
    return grp


# ------------------------------
# 5. Przygotowanie katalogu na HTML
# ------------------------------
out_dir = 'HTML_loty'
os.makedirs(out_dir, exist_ok=True)

phase_colors = {
    'Lower Climb':'blue','Upper Climb':'cyan',
    'Lower Cruise':'green','Upper Cruise':'darkgreen',
    'Upper Descent':'orange','Lower Descent':'red',
    'Unknown':'gray'
}

# ------------------------------
# 6. Pętla po segmentach: detekcja faz i rysowanie mapy
# ------------------------------
for ident, seg_idx, seg in all_segments:
    if len(seg)<2:
        continue
    seg = detect_phases(seg)
    # kolejny numer segmentu wewnątrz danego ident
    seg['seg_phase'] = (seg['phase']!=seg['phase'].shift()).cumsum()
    # inicjalizacja mapy na pierwszy punkt
    start = [seg.iloc[0].lat, seg.iloc[0].lon]
    m = folium.Map(location=start, zoom_start=6)
    for _, sub in seg.groupby('seg_phase'):
        pts = sub[['lat','lon']].values.tolist()
        ph = sub.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')
        folium.PolyLine(pts, color=clr, weight=3, opacity=0.8).add_to(m)
        # marker start/end
        for label, row in [('start', sub.iloc[0]), ('end', sub.iloc[-1])]:
            popup = Popup(
                f"{ph} {label}<br>"
                f"time: {row['clock']}<br>"
                f"alt: {row['alt']} ft<br>"
                f"gs: {row['gs']} kt<br>"
                f"roc: {row['vertRate']} fpm",
                max_width=200
            )
            icon = '●' if label=='start' else '◆'
            folium.Marker(
                [row.lat, row.lon],
                popup=popup,
                icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>{icon}</div>")
            ).add_to(m)

    filename = f"{ident}_seg{seg_idx}.html"
    m.save(os.path.join(out_dir, filename))
    print(f"Saved: {filename}")

print(f"\nWygenerowano {len(os.listdir(out_dir))} plików HTML w katalogu ./{out_dir}/")


<ipython-input-2-2bdfc91e1f8c>:119: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Lower Climb' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Lower Climb"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna('Unknown')
<ipython-input-2-2bdfc91e1f8c>:135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Upper Cruise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Upper Cruise"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version

Saved: ABB26_seg1.html
Saved: ABB26_seg2.html
Saved: ABB26_seg3.html
Saved: ABB26_seg4.html
Saved: ABY749_seg1.html
Saved: AEE868_seg1.html
Saved: AEE868_seg2.html
Saved: AEE869_seg1.html
Saved: AEE869_seg2.html
Saved: AEE873_seg1.html


<ipython-input-2-2bdfc91e1f8c>:135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Upper Cruise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Upper Cruise"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna('Unknown')
<ipython-input-2-2bdfc91e1f8c>:126: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Lower Descent' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Lower Descent"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future ver

Saved: AFR1046_seg1.html
Saved: AFR1046_seg2.html
Saved: AFR1047_seg1.html
Saved: AFR1247_seg1.html
Saved: AFR1346_seg1.html
Saved: AFR1346_seg2.html
Saved: AFR1347_seg1.html
Saved: AUA597_seg1.html
Saved: AUA598_seg1.html


<ipython-input-2-2bdfc91e1f8c>:119: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Lower Climb' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Lower Climb"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna('Unknown')
<ipython-input-2-2bdfc91e1f8c>:135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Upper Cruise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Upper Cruise"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version

...

Saved: WZZ2051_seg2.html
Saved: WZZ2052_seg1.html
Saved: WZZ2063_seg1.html
Saved: WZZ2063_seg2.html
Saved: WZZ2064_seg1.html
Saved: WZZ2071_seg1.html
Saved: WZZ2075_seg1.html
Saved: WZZ2075_seg2.html


<ipython-input-2-2bdfc91e1f8c>:119: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Lower Climb' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Lower Climb"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna('Unknown')
<ipython-input-2-2bdfc91e1f8c>:128: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Upper Descent' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Upper Descent"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future versi

Saved: WZZ2093_seg1.html
Saved: WZZ2094_seg1.html
Saved: WZZ2094_seg2.html
Saved: WZZ2094_seg3.html
Saved: WZZ2097_seg1.html
Saved: WZZ2097_seg2.html
Saved: WZZ2098_seg1.html
Saved: WZZ2467_seg1.html
Saved: WZZ7941_seg1.html


<ipython-input-2-2bdfc91e1f8c>:135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Upper Cruise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Upper Cruise"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna('Unknown')
<ipython-input-2-2bdfc91e1f8c>:135: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Upper Cruise' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  grp.at[i, 'phase'] = "Upper Cruise"
<ipython-input-2-2bdfc91e1f8c>:140: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future versi

Saved: WZZ7941_seg2.html
Saved: WZZ92_seg1.html

Wygenerowano 1051 plików HTML w katalogu ./HTML_loty/


In [ ]:
import os
import pandas as pd
import folium
from folium.plugins import AntPath


# Utwórz folder 'HTML' do przechowywania map, jeśli nie istnieje
output_dir = 'HTML'
os.makedirs(output_dir, exist_ok=True)

# Iteruj przez każdą grupę (samolot)
for ident, group in grouped:
    # Sprawdź, czy są dostępne współrzędne
    if group[['lat', 'lon']].isnull().any().any():
        print(f"Pomijam {ident} z powodu brakujących danych geograficznych.")
        continue

    # Utwórz mapę skoncentrowaną na pierwszym punkcie lotu
    start_location = [group.iloc[0]['lat'], group.iloc[0]['lon']]
    m = folium.Map(location=start_location, zoom_start=6)

    # Dodaj ścieżkę lotu jako linię
    locations = group[['lat', 'lon']].values.tolist()
    AntPath(locations=locations, color='blue', weight=2.5, opacity=0.8).add_to(m)

    # Dodaj znaczniki dla punktów początkowego i końcowego
    folium.Marker(location=locations[0], popup='Start', icon=folium.Icon(color='green')).add_to(m)
    folium.Marker(location=locations[-1], popup='Koniec', icon=folium.Icon(color='red')).add_to(m)

    # Zapisz mapę do pliku HTML
    filename = f"{ident}.html"
    filepath = os.path.join(output_dir, filename)
    m.save(filepath)
    print(f"Zapisano mapę dla {ident} jako {filepath}")


ValueError: Expected two (lat, lon) values for location, instead got: '52.13865'.

In [ ]:
# 13) Przygotuj folder HTML
out_dir = 'HTML'
os.makedirs(out_dir, exist_ok=True)

# 14) Generowanie map dla każdego ident
for ident, grp in df_phases.groupby('ident'):
    # Usuń wiersze bez współrzędnych
    grp_clean = grp.dropna(subset=['lat','lon'])
    if len(grp_clean) < 2:
        # za mało punktów do narysowania
        continue

    # Wybierz pierwszy poprawny punkt jako środek mapy
    start_row = grp_clean.iloc[0]
    start_loc = [start_row['lat'], start_row['lon']]

    # Zainicjuj mapę
    m = folium.Map(location=start_loc, zoom_start=6)

    # Podziel trasę na segmenty faz
    grp_clean = grp_clean.sort_values('clock').reset_index(drop=True)
    grp_clean['seg_id'] = (grp_clean['phase'] != grp_clean['phase'].shift()).cumsum()

    for _, seg in grp_clean.groupby('seg_id'):
        pts = seg[['lat','lon']].values.tolist()
        ph  = seg.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')

        # Narysuj odcinek
        folium.PolyLine(pts, color=clr, weight=4, opacity=0.8).add_to(m)

        # Dodaj marker startu fazy
        r0 = seg.iloc[0]
        folium.Marker(
            [r0['lat'], r0['lon']],
            popup=f"{ph} start<br>time: {r0['clock']}<br>alt: {r0['alt']}<br>gs: {r0['gs']}",
            icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>●</div>")
        ).add_to(m)

        # Dodaj marker końca fazy
        r1 = seg.iloc[-1]
        folium.Marker(
            [r1['lat'], r1['lon']],
            popup=f"{ph} end<br>time: {r1['clock']}<br>alt: {r1['alt']}<br>gs: {r1['gs']}",
            icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>◆</div>")
        ).add_to(m)

    # Zapisz mapę do HTML
    filepath = os.path.join(out_dir, f"{ident}.html")
    m.save(filepath)
    print(f"Saved map for {ident}: {filepath}")


Saved map for LOT3510: HTML/LOT3510.html
Saved map for LOT3802: HTML/LOT3802.html
Saved map for LOT3803: HTML/LOT3803.html
Saved map for LOT3804: HTML/LOT3804.html
Saved map for LOT3816: HTML/LOT3816.html
Saved map for LOT3827: HTML/LOT3827.html
Saved map for LOT3828: HTML/LOT3828.html
Saved map for LOT3837: HTML/LOT3837.html
Saved map for LOT3838: HTML/LOT3838.html
Saved map for LOT3844: HTML/LOT3844.html
Saved map for LOT3848: HTML/LOT3848.html
Saved map for LOT3849: HTML/LOT3849.html
Saved map for LOT3850: HTML/LOT3850.html
Saved map for LOT3851: HTML/LOT3851.html
Saved map for LOT3859: HTML/LOT3859.html
Saved map for LOT3879: HTML/LOT3879.html
Saved map for LOT3880: HTML/LOT3880.html
Saved map for LOT3884: HTML/LOT3884.html
Saved map for LOT3886: HTML/LOT3886.html
Saved map for LOT3906: HTML/LOT3906.html
Saved map for LOT3910: HTML/LOT3910.html
Saved map for LOT3911: HTML/LOT3911.html
Saved map for LOT3924: HTML/LOT3924.html
Saved map for LOT3931: HTML/LOT3931.html
Saved map for LO

In [ ]:
import os
import pandas as pd
import numpy as np
import folium
from folium import Popup, DivIcon

# ------------------------------
# Wczytaj dane z pliku JSON
# ------------------------------
df = pd.read_json('all_flights_2.json', orient='records')
df['clock'] = pd.to_datetime(df['clock'], unit='s', errors='coerce')

df = df.dropna(subset=['clock','lat','lon','alt','gs','vertRate','aircrafttype'])
df = df.sort_values(['ident','clock']).reset_index(drop=True)

# ------------------------------
# Mach/GS wartości referencyjne per typ
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}

# ------------------------------
# Funkcja wykrywająca fazy
# ------------------------------
def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    grp['phase'] = pd.Series(dtype='object')

    ac_type = grp['aircrafttype'].iloc[0]
    params = aircraft_speeds.get(ac_type)
    if not params:
        return grp  # Brak danych - pomiń

    FL = params["FL_CRUISE"]
    Vcl1 = params["Vcl1"]
    Vcr2 = params["Vcr2"]
    Vdes1 = params["Vdes1"]
    Mcl = params["Mcl"]
    Mcr = params["Mcr"]
    Mdes = params["Mdes"]

    def get_mach_limit(mach):
        return mach * 574  # a(h) ~ 574 kt @ FL370

    for i, row in grp.iterrows():
        alt = row['alt']
        gs = row['gs']
        roc = row['vertRate']

        if 100 <= alt <= FL - 4000 and roc >= 250:
            if gs <= Vcl1:
                grp.at[i, 'phase'] = "Lower Climb"
            else:
                grp.at[i, 'phase'] = "Upper Climb"
        elif abs(alt - FL) <= 3000 and abs(roc) < 250:
            if gs <= Vcr2:
                grp.at[i, 'phase'] = "Lower Cruise"
            else:
                grp.at[i, 'phase'] = "Upper Cruise"
        elif 1000 <= alt <= FL - 2000 and roc <= -250:
            if gs <= Vdes1 or (alt < 10000 and gs <= 250):
                grp.at[i, 'phase'] = "Lower Descent"
            else:
                grp.at[i, 'phase'] = "Upper Descent"

    grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill')
    return grp

# ------------------------------
# Zastosuj detekcję faz
# ------------------------------
df_ph = df.groupby('ident', group_keys=False).apply(detect_phases)
df_ph = df_ph.reset_index(drop=True)

# ------------------------------
# Kolory dla faz
# ------------------------------
phase_colors = {
    'Lower Climb':    'blue',
    'Upper Climb':    'cyan',
    'Lower Cruise':   'green',
    'Upper Cruise':   'darkgreen',
    'Upper Descent':  'orange',
    'Lower Descent':  'red',
}

# ------------------------------
# Generowanie HTML map
# ------------------------------
out_dir = 'HTML'
os.makedirs(out_dir, exist_ok=True)

for ident, grp in df_ph.groupby('ident'):
    grp = grp.dropna(subset=['lat','lon'])
    if len(grp) < 2:
        continue

    start = [grp.iloc[0]['lat'], grp.iloc[0]['lon']]
    m = folium.Map(location=start, zoom_start=6)

    grp['seg'] = (grp['phase'] != grp['phase'].shift()).cumsum()
    for _, seg in grp.groupby('seg'):
        pts = seg[['lat','lon']].values.tolist()
        ph = seg.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')
        folium.PolyLine(pts, color=clr, weight=3, opacity=0.8).add_to(m)
        for label, row in [('start', seg.iloc[0]), ('end', seg.iloc[-1])]:
            popup = Popup(
                f"{ph} {label}<br>"
                f"time: {row['clock']}<br>"
                f"alt: {row['alt']} ft<br>"
                f"gs: {row['gs']} kt<br>"
                f"roc: {row['vertRate']} fpm",
                max_width=200
            )
            icon_char = '●' if label == 'start' else '◆'
            folium.Marker(
                [row['lat'], row['lon']],
                popup=popup,
                icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>{icon_char}</div>")
            ).add_to(m)

    m.save(f"{out_dir}/{ident}.html")
    print(f"Saved map for {ident}")

print(f"\nGenerated {len(os.listdir(out_dir))} HTML files in ./{out_dir}/")


FloatingPointError: overflow encountered in multiply

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ------------------------------
# Wczytaj dane z pliku JSON
# ------------------------------
df = pd.read_json('all_flights.json', orient='records')
df['clock'] = pd.to_datetime(df['clock'], unit='s', errors='coerce')
df = df.dropna(subset=['clock','lat','lon','alt','gs','vertRate','aircrafttype'])
df = df.sort_values(['ident','clock']).reset_index(drop=True)
# ------------------------------
# Mach/GS wartości referencyjne per typ
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}
# ------------------------------
# Funkcja wykrywająca fazy
# ------------------------------
def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    grp['phase'] = pd.Series(dtype='object')
    ac_type = grp['aircrafttype'].iloc[0]
    params = aircraft_speeds.get(ac_type)
    if not params:
        return grp  # Brak danych - pomiń
    FL = params["FL_CRUISE"]
    Vcl1 = params["Vcl1"]
    Vcr2 = params["Vcr2"]
    Vdes1 = params["Vdes1"]
    Mcl = params["Mcl"]
    Mcr = params["Mcr"]
    Mdes = params["Mdes"]
    def get_mach_limit(mach):
        return mach * 574  # a(h) ~ 574 kt @ FL370
    for i, row in grp.iterrows():
        alt = row['alt']
        gs = row['gs']
        roc = row['vertRate']
        if 10 <= alt <= FL - 2000 and roc >= 150:
            if gs <= Vcl1:
                grp.at[i, 'phase'] = "Lower Climb"
            else:
                grp.at[i, 'phase'] = "Upper Climb"
        elif abs(alt - FL) <= 1500 and abs(roc) < 250:
            if gs <= Vcr2:
                grp.at[i, 'phase'] = "Lower Cruise"
            else:
                grp.at[i, 'phase'] = "Upper Cruise"
        elif 100 <= alt <= FL - 2000 and roc <= -150:
            if gs <= Vdes1 or (alt < 10000 and gs <= 250):

                grp.at[i, 'phase'] = "Lower Descent"
            else:
                grp.at[i, 'phase'] = "Upper Descent"
    grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill')
    return grp
# ------------------------------
# Zastosuj detekcję faz
# ------------------------------
df_ph = df.groupby('ident', group_keys=False).apply(detect_phases)
df_ph = df_ph.reset_index(drop=True)


In [ ]:
aircraft_parameters = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}

  Using cached numpy-1.26.4-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.2.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (19 kB)
INFO: pip is looking at multiple versions of pybada to determine which version is compatible with other requirements. This could take a while.
  Using cached pybada-0.1.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached pybada-0.1.0-py3-none-any.whl.metadata (2.1 kB)
ERROR: Cannot install numpy==1.26.4, pandas==2.2.2, pybada==0.1.0, pybada==0.1.1 and pybada==0.1.2 because these package versions have conflicting dependencies.

The conflict is caused by:
    The user requested numpy==1.26.4
    pandas 2.2.2 depends on numpy>=1.23.2; python_version == "3.11"
    pybada 0.1.2 depends on numpy==1.26.1
    The user requested numpy==1.26.4
    pandas 2.2.2 depends on numpy>=1.23.2; python_version == "3.11"
    pybada 0.1.1 depends on numpy==1.26.1
    The user requested numpy==1.26.4
   

In [ ]:
import os
import pandas as pd
import numpy as np
import folium
from folium import Popup, DivIcon

# ------------------------------
# Wczytaj dane z pliku JSON
# ------------------------------
df = pd.read_json('all_flights_2.json', orient='records')
df['clock'] = pd.to_datetime(df['clock'], unit='s', errors='coerce')

df = df.dropna(subset=['clock','lat','lon','alt','gs','vertRate','aircrafttype'])
df = df.sort_values(['ident','clock']).reset_index(drop=True)

# ------------------------------
# Mach/GS wartości referencyjne per typ
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}

# ------------------------------
# Funkcja wykrywająca fazy
# ------------------------------
def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    grp['phase'] = pd.Series(dtype='object')

    ac_type = grp['aircrafttype'].iloc[0]
    params = aircraft_speeds.get(ac_type)
    if not params:
        return grp  # Brak danych - pomiń

    FL = params["FL_CRUISE"]
    Vcl1 = params["Vcl1"]
    Vcr2 = params["Vcr2"]
    Vdes1 = params["Vdes1"]
    Mcl = params["Mcl"]
    Mcr = params["Mcr"]
    Mdes = params["Mdes"]

    def get_mach_limit(mach):
        return mach * 574  # a(h) ~ 574 kt @ FL370

    for i, row in grp.iterrows():
        alt = row['alt']
        gs = row['gs']
        roc = row['vertRate']

        if 10 <= alt <= FL - 4000 and roc >= 150: ## trzeba dodac zalozenie ze pierwszy punkt climbu musi zaczynac sie z obszaru lotniska
        ## Dodatkowo warto zaznaczyc ze pierwszy i ostatni punkt climbu nie moze byc super daleko od siebie
            if gs <= Vcl1:
                grp.at[i, 'phase'] = "Lower Climb"
            else:                                     ## dawniej elif gs >= get_mach_limit(Mcl): ale powodowalo ze nie uzupelnialo danych
                grp.at[i, 'phase'] = "Upper Climb"
        elif abs(alt - FL) <= 4000 and abs(roc) < 150:
            if gs <= Vcr2:
                grp.at[i, 'phase'] = "Lower Cruise"
            else:                                   ## Dawniej elif gs >= get_mach_limit(Mcr)
                grp.at[i, 'phase'] = "Upper Cruise"
        elif 100 <= alt <= FL - 4000 and roc <= -150:
            if gs <= Vdes1 or (alt < 10000 and gs <= 250):
              ## trzeba dodac zalozenie ze ostatni punkt descentu musi byc w obszarze lotniska
              ## Dodatkowo warto zaznaczyc ze pierwszy i ostatni punkt climbu nie moze byc super daleko od siebie
                grp.at[i, 'phase'] = "Lower Descent"
            else:                                     ## dawniej  elif gs >= get_mach_limit(Mdes): ale powodowalo ze nie uzupelnialo danych
                grp.at[i, 'phase'] = "Upper Descent"

    grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill')
    return grp

# ------------------------------
# Zastosuj detekcję faz
# ------------------------------
df_ph = df.groupby('ident', group_keys=False).apply(detect_phases)
df_ph = df_ph.reset_index(drop=True)

# ------------------------------
# Kolory dla faz
# ------------------------------
phase_colors = {
    'Lower Climb':    'blue',
    'Upper Climb':    'cyan',
    'Lower Cruise':   'green',
    'Upper Cruise':   'darkgreen',
    'Upper Descent':  'orange',
    'Lower Descent':  'red',
}

# ------------------------------
# Generowanie HTML map
# ------------------------------
out_dir = 'HTML'
os.makedirs(out_dir, exist_ok=True)

for ident, grp in df_ph.groupby('ident'):
    grp = grp.dropna(subset=['lat','lon'])
    if len(grp) < 2:
        continue

    start = [grp.iloc[0]['lat'], grp.iloc[0]['lon']]
    m = folium.Map(location=start, zoom_start=6)

    grp['seg'] = (grp['phase'] != grp['phase'].shift()).cumsum()
    for _, seg in grp.groupby('seg'):
        pts = seg[['lat','lon']].values.tolist()
        ph = seg.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')
        folium.PolyLine(pts, color=clr, weight=3, opacity=0.8).add_to(m)
        for label, row in [('start', seg.iloc[0]), ('end', seg.iloc[-1])]:
            popup = Popup(
                f"{ph} {label}<br>"
                f"time: {row['clock']}<br>"
                f"alt: {row['alt']} ft<br>"
                f"gs: {row['gs']} kt<br>"
                f"roc: {row['vertRate']} fpm",
                max_width=200
            )
            icon_char = '●' if label == 'start' else '◆'
            folium.Marker(
                [row['lat'], row['lon']],
                popup=popup,
                icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>{icon_char}</div>")
            ).add_to(m)

    m.save(f"{out_dir}/{ident}.html")
    print(f"Saved map for {ident}")

print(f"\nGenerated {len(os.listdir(out_dir))} HTML files in ./{out_dir}/")


FloatingPointError: overflow encountered in multiply

In [ ]:
import pandas as pd

# Wczytanie tabeli z pliku tekstowego
with open('dada.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

# Filtracja linii z danymi (pomijamy separator '+' i puste linie)
data_lines = [l for l in lines if l.startswith('|')]

# Pierwsza linia to nagłówek
header = [h.strip() for h in data_lines[0].strip().strip('|').split('|')]

# Pozostałe linie to wiersze danych
records = []
for line in data_lines[1:]:
    values = [v.strip() for v in line.strip().strip('|').split('|')]
    records.append(values)

# Utworzenie DataFrame
df = pd.DataFrame(records, columns=header)

# Zastąpienie wartości 'NULL' pustymi ciągami
df = df.replace('NULL', '')

# Zapis do pliku CSV
output_path = 'output.csv'
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"Zapisano plik CSV: {output_path}")


Zapisano plik CSV: output.csv


In [ ]:
import pandas as pd

# 1. Wczytaj dane
df = pd.read_csv('output.csv', encoding='utf-8')

# 2. Kolumny do sprawdzenia
cols = [
    'Upper Cruise Speed',
    'Lower Cruise Speed',
    'Upper Climb Speed',
    'Lower Climb Speed',
    'Climb Mach Speed',
    'Upper Descent Speed',
    'Lower Descent Speed',
    'Descent Mach Speed'
]

# 3. Usuń wiersze, w których WE WSZYSTKICH tych kolumn są wartości NaN
df_clean = df.dropna(subset=cols, how='all')  # how='all' → tylko gdy wszystkie są NA :contentReference[oaicite:0]{index=0}

# 4. Zapisz wynik
df_clean.to_csv('output_clean.csv', index=False, encoding='utf-8')

print(f"Oryginalnych wierszy: {len(df)}")
print(f"Po usunięciu (all-null): {len(df_clean)}")


Oryginalnych wierszy: 500
Po usunięciu (all-null): 394


In [ ]:
# Jeśli nie masz zainstalowanego pyarrow:
!pip install pyarrow

import pandas as pd
import glob
import os

# 1. Ścieżka do katalogu z plikami .parquet
#    Możesz to zamienić na ścieżkę do Google Drive, jeśli montujesz go w Colabie:
parquet_dir = "."

# 2. Znajdź wszystkie pliki .parquet w katalogu (i ewentualnie podkatalogach)
#    Jeśli masz partycjonowane foldery YYYY/MM/DD możesz użyć np. "/content/data/flights/**/*.parquet"
files = glob.glob(os.path.join(parquet_dir, "**", "*.parquet"), recursive=True)

# 3. Wczytaj i połącz
#    Uwaga: przy bardzo dużej liczbie plików może chwilę potrwać
df_list = [pd.read_parquet(f, engine="pyarrow") for f in files]
df = pd.concat(df_list, ignore_index=True)

# 4. Gotowe – masz pandas DataFrame 'df'
print(f"Wczytano {len(files)} plików Parquet, łącznie {df.shape[0]} wierszy i {df.shape[1]} kolumn.")
df.head()


Wczytano 1 plików Parquet, łącznie 6053 wierszy i 12 kolumn.


,flight_id,ident,clock,lat,lon,alt,speed,gs,vertRate,orig,dest,aircrafttype
0,0,ABB26,NaT,52.26950,21.01967,8050.0,None,292.0,1920.0,EPWA,KJFK,A339
1,0,ABB26,NaT,52.35814,20.88028,10425.0,None,315.0,768.0,EPWA,KJFK,A339
2,0,ABB26,NaT,52.42708,20.77156,11125.0,None,360.0,640.0,EPWA,KJFK,A339
3,0,ABB26,NaT,52.51630,20.62325,12900.0,None,379.0,1536.0,EPWA,KJFK,A339
4,0,ABB26,NaT,54.93860,15.52433,36000.0,None,456.0,0.0,EPWA,KJFK,A339


In [ ]:
# jeśli nie masz folium:
# !pip install folium

import os
import glob
import pandas as pd
import numpy as np
import folium
from folium import Popup, DivIcon

# ------------------------------
# 1. Wczytanie danych z Parquet
# ------------------------------
parquet_dir = "."   # dostosuj do swojej ścieżki
files = glob.glob(os.path.join(parquet_dir, "**", "*.parquet"), recursive=True)

# łączymy wszystkie pliki w jeden DataFrame
df_list = [pd.read_parquet(f, engine="pyarrow") for f in files]
df = pd.concat(df_list, ignore_index=True)

# Konwersja clock i podstawowe filtrowanie
df['clock'] = pd.to_datetime(df['clock'], errors='coerce')
df = df.dropna(subset=['clock','lat','lon','alt','gs','vertRate','aircrafttype'])
df = df.sort_values(['ident','clock']).reset_index(drop=True)

# ------------------------------
# 2. Definicja referencyjnych prędkości
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}

# ------------------------------
# 3. Funkcja detekcji faz
# ------------------------------
def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    grp['phase'] = pd.NA

    ac_type = grp['aircrafttype'].iloc[0]
    params = aircraft_speeds.get(ac_type)
    if params is None:
        return grp  # brak parametrów dla tego typu

    FL = params["FL_CRUISE"]
    Vcl1 = params["Vcl1"]
    Vcr2 = params["Vcr2"]
    Vdes1 = params["Vdes1"]
    # helper do mach→kt
    def mach_to_kt(m): return m * 574

    for i, row in grp.iterrows():
        alt = row['alt']
        gs  = row['gs']
        roc = row['vertRate']

        # Climb
        if 10 <= alt <= FL-4000 and roc >= 150:
            grp.at[i,'phase'] = "Lower Climb" if gs <= Vcl1 else "Upper Climb"
        # Cruise
        elif abs(alt-FL) <= 4000 and abs(roc) < 150:
            grp.at[i,'phase'] = "Lower Cruise" if gs <= Vcr2 else "Upper Cruise"
        # Descent
        elif 100 <= alt <= FL-4000 and roc <= -150:
            if gs <= Vdes1 or (alt<10000 and gs<=250):
                grp.at[i,'phase'] = "Lower Descent"
            else:
                grp.at[i,'phase'] = "Upper Descent"

    # forward/backward fill braków
    grp['phase'] = grp['phase'].ffill().bfill()
    return grp

# ------------------------------
# 4. Zastosowanie detekcji faz
# ------------------------------
df_ph = df.groupby('ident', group_keys=False).apply(detect_phases).reset_index(drop=True)

# ------------------------------
# 5. Przygotowanie kolorów
# ------------------------------
phase_colors = {
    'Lower Climb':    'blue',
    'Upper Climb':    'cyan',
    'Lower Cruise':   'green',
    'Upper Cruise':   'darkgreen',
    'Upper Descent':  'orange',
    'Lower Descent':  'red',
}

# ------------------------------
# 6. Generowanie map HTML
# ------------------------------
out_dir = 'HTML'
os.makedirs(out_dir, exist_ok=True)

for ident, grp in df_ph.groupby('ident'):
    grp = grp.dropna(subset=['lat','lon'])
    if len(grp) < 2:
        continue

    start = [grp.iloc[0]['lat'], grp.iloc[0]['lon']]
    m = folium.Map(location=start, zoom_start=6)

    # segmentacja po zmianie fazy
    grp['seg'] = (grp['phase'] != grp['phase'].shift()).cumsum()
    for _, seg in grp.groupby('seg'):
        pts = seg[['lat','lon']].values.tolist()
        ph = seg.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')

        folium.PolyLine(pts, color=clr, weight=3, opacity=0.8).add_to(m)
        # znaczniki start/koniec odcinka
        for label, row in [('start', seg.iloc[0]), ('end', seg.iloc[-1])]:
            popup = Popup(
                f"{ph} {label}<br>"
                f"time: {row['clock']}<br>"
                f"alt: {row['alt']} ft<br>"
                f"gs: {row['gs']} kt<br>"
                f"roc: {row['vertRate']} fpm",
                max_width=200
            )
            icon_char = '●' if label=='start' else '◆'
            folium.Marker(
                [row['lat'], row['lon']],
                popup=popup,
                icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>{icon_char}</div>")
            ).add_to(m)

    m.save(f"{out_dir}/{ident}.html")
    print(f"Saved map for {ident}")

print(f"\nGenerated {len(os.listdir(out_dir))} HTML files in './{out_dir}/'")



Generated 0 HTML files in './HTML/'


<ipython-input-3-d3c8ce20ded2>:106: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_ph = df.groupby('ident', group_keys=False).apply(detect_phases).reset_index(drop=True)


In [ ]:

import json
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Funkcja obliczająca odległość w kilometrach między dwoma punktami geograficznymi
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Promień Ziemi w km
    phi1, phi2 = radians(lat1), radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)
    a = sin(delta_phi / 2)**2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# Wczytanie danych
with open('all_flights_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Filtrowanie tylko pozycji + usunięcie wierszy z brakami
df = df[df['type'] == 'position']
df_clean = df.dropna(subset=['lat', 'lon', 'alt', 'gs', 'vertRate', 'aircrafttype','clock', 'id', 'orig', 'dest', 'ident']).copy()

# Konwersje typów
df_clean['lat'] = df_clean['lat'].astype(float)
df_clean['lon'] = df_clean['lon'].astype(float)
df_clean['alt'] = df_clean['alt'].astype(float)
df_clean['gs'] = df_clean['gs'].astype(float)
df_clean['clock'] = df_clean['clock'].astype(int)

# Sortowanie danych
df_sorted = df_clean.sort_values(['ident', 'clock'])
grouped = df_sorted.groupby('ident')

# Segmentacja
for ident, group in grouped:
    group = group.reset_index(drop=True)
    segments = []
    current_segment = [group.loc[0]]

    for i in range(1, len(group)):
        lat1, lon1 = group.loc[i - 1, ['lat', 'lon']]
        lat2, lon2 = group.loc[i, ['lat', 'lon']]
        alt1 = group.loc[i - 1, 'alt']
        time1 = group.loc[i - 1, 'clock']
        time2 = group.loc[i, 'clock']
        distance = haversine(lat1, lon1, lat2, lon2)
        time_diff = time2 - time1

        # Ustalanie progów zależnych od wysokości
        if alt1 > 30000:
            distance_threshold = 100  # km
            time_threshold = 600     # sekundy
        elif alt1 > 20000:
            distance_threshold = 70
            time_threshold = 500
        elif alt1 > 10000:
            distance_threshold = 50
            time_threshold = 400
        else:
            distance_threshold = 40
            time_threshold = 300

        # Sprawdzenie, czy rozpocząć nowy segment
        if distance > distance_threshold or time_diff > time_threshold:
            segments.append(pd.DataFrame(current_segment))
            current_segment = [group.loc[i]]
        else:
            current_segment.append(group.loc[i])

    # Dodaj ostatni segment
    if current_segment:
        segments.append(pd.DataFrame(current_segment))

    # Wyświetl
    for idx, segment in enumerate(segments, start=1):
        print(f"\n{'-'*50}\n")
        print(f"Przelot: {ident} | Segment: {idx}")
        print(segment[['clock', 'aircrafttype', 'ident',  'orig', 'dest', 'lat', 'lon', 'alt', 'gs', 'vertRate' ]])

    print(f"Znaleziono {len(all_segments)} segmentów w locie.")


Strumieniowane dane wyjściowe obcięte do 5000 ostatnich wierszy.
Przelot: WZZ1241 | Segment: 1
        clock aircrafttype    ident  orig  dest       lat       lon      alt  \
0  1726595933         A320  WZZ1241  EPKT  LGAV  50.47527  19.10877   1650.0   
1  1726595949         A320  WZZ1241  EPKT  LGAV  50.47476  19.12532   2250.0   
2  1726595965         A320  WZZ1241  EPKT  LGAV  50.47445  19.14538   2600.0   
3  1726595997         A320  WZZ1241  EPKT  LGAV  50.46002  19.18516   3225.0   
4  1726596168         A320  WZZ1241  EPKT  LGAV  50.24360  19.25770  11425.0   
5  1726596348         A320  WZZ1241  EPKT  LGAV  49.95607  19.35473  18775.0   
6  1726596378         A320  WZZ1241  EPKT  LGAV  49.90453  19.37200  19750.0   
7  1726596408         A320  WZZ1241  EPKT  LGAV  49.85121  19.38976  20775.0   
8  1726596461         A320  WZZ1241  EPKT  LGAV  49.75841  19.41429  22350.0   

      gs vertRate  
0  149.0     3392  
1  154.0     1216  
2  177.0     1216  
3  240.0     1472  
4  3

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

import folium
from folium import Popup
from folium.features import DivIcon

# ------------------------------
# 1. Funkcja obliczająca odległość (haversine)
# ------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Promień Ziemi w km
    phi1, phi2 = radians(lat1), radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)
    a = sin(delta_phi / 2)**2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# ------------------------------
# 2. Profile prędkości/Mach dla typów samolotów
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}

# ------------------------------
# 3. Funkcja wykrywająca fazy lotu
# ------------------------------
def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    # --- dodajemy konwersję na float, żeby uniknąć TypeError ---
    grp[['alt', 'gs', 'vertRate']] = grp[['alt', 'gs', 'vertRate']].astype(float)
    grp['phase'] = pd.NA

    ac_type = grp['aircrafttype'].iloc[0]
    params = aircraft_speeds.get(ac_type)
    if not params:
        grp['phase'] = "Unknown"
        return grp

    FL = params["FL_CRUISE"]
    Vcl1, Vcr2, Vdes1 = params["Vcl1"], params["Vcr2"], params["Vdes1"]

    for i, row in grp.iterrows():
        alt, gs, roc = row['alt'], row['gs'], row['vertRate']
        # climb
        if 10 <= alt <= FL - 2000 and roc >= 150:
            grp.at[i, 'phase'] = "Lower Climb" if gs <= Vcl1 else "Upper Climb"
        # cruise
        elif abs(alt - FL) <= 1500 and abs(roc) < 250:
            grp.at[i, 'phase'] = "Lower Cruise" if gs <= Vcr2 else "Upper Cruise"
        # descent
        elif 100 <= alt <= FL - 2000 and roc <= -150:
            if gs <= Vdes1 or (alt < 10000 and gs <= 250):
                grp.at[i, 'phase'] = "Lower Descent"
            else:
                grp.at[i, 'phase'] = "Upper Descent"
        else:
            grp.at[i, 'phase'] = pd.NA

    grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
    return grp


# ------------------------------
# 4. Wczytanie i wstępne czyszczenie danych
# ------------------------------
with open('all_flights_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
df = df[df['type'] == 'position'].dropna(subset=['lat','lon','alt','gs','vertRate','aircrafttype','clock','id','orig','dest','ident'])
df = df.astype({'lat': float, 'lon': float, 'alt': float, 'gs': float, 'clock': int})
df = df.sort_values(['ident','clock'])

# ------------------------------
# 5. Segmentacja lotów
# ------------------------------
all_segments = []  # tu będziemy zbierać (ident, idx, df_segment)
for ident, group in df.groupby('ident', group_keys=False):
    group = group.reset_index(drop=True)
    segments = []
    current = [group.loc[0]]
    for i in range(1, len(group)):
        prev, curr = group.loc[i-1], group.loc[i]
        dist = haversine(prev.lat, prev.lon, curr.lat, curr.lon)
        dt = curr.clock - prev.clock
        alt1 = prev.alt
        # progi zależne od wysokości
        if alt1 > 30000:
            d_thresh, t_thresh = 100, 600
        elif alt1 > 20000:
            d_thresh, t_thresh = 70, 500
        elif alt1 > 10000:
            d_thresh, t_thresh = 50, 400
        else:
            d_thresh, t_thresh = 40, 300
        if dist > d_thresh or dt > t_thresh:
            segments.append(pd.DataFrame(current))
            current = [curr]
        else:
            current.append(curr)
    if current:
        segments.append(pd.DataFrame(current))
    # dodajemy do wspólnej listy
    for idx, seg in enumerate(segments, start=1):
        all_segments.append((ident, idx, seg))

print(f"Znaleziono {len(all_segments)} segmentów w locie.")

# ------------------------------
# 6. Generowanie HTML z mapami
# ------------------------------
out_dir = 'WYSWIETLONE_LOTY'
os.makedirs(out_dir, exist_ok=True)
phase_colors = {
    'Lower Climb':'blue','Upper Climb':'cyan',
    'Lower Cruise':'green','Upper Cruise':'darkgreen',
    'Upper Descent':'orange','Lower Descent':'red',
    'Unknown':'gray'
}

for ident, seg_idx, seg in all_segments:
    if len(seg) < 2:
        continue
    seg = detect_phases(seg)
    seg['seg_phase'] = (seg['phase'] != seg['phase'].shift()).cumsum()
    start = [seg.iloc[0].lat, seg.iloc[0].lon]
    m = folium.Map(location=start, zoom_start=6)
    for _, sub in seg.groupby('seg_phase'):
        pts = sub[['lat','lon']].values.tolist()
        ph = sub.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')
        folium.PolyLine(pts, color=clr, weight=3, opacity=0.8).add_to(m)
        # markery
        for label, row in [('start', sub.iloc[0]), ('end', sub.iloc[-1])]:
            popup = Popup(
                f"{ph} {label}<br>"
                f"time: {row['clock']}<br>"
                f"alt: {row['alt']} ft<br>"
                f"gs: {row['gs']} kt<br>"
                f"roc: {row['vertRate']} fpm",
                max_width=200
            )
            icon = '●' if label == 'start' else '◆'
            folium.Marker(
                [row.lat, row.lon],
                popup=popup,
                icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>{icon}</div>")
            ).add_to(m)
    filename = f"{ident}_seg{seg_idx}.html"
    m.save(os.path.join(out_dir, filename))
    print(f"Saved: {filename}")

print(f"\nWygenerowano {len(os.listdir(out_dir))} plików HTML w katalogu ./{out_dir}/")


Znaleziono 988 segmentów w locie.
Saved: ABB26_seg1.html
Saved: ABB26_seg2.html
Saved: ABB26_seg3.html
Saved: ABB26_seg4.html
Saved: ABY749_seg1.html
Saved: AEE868_seg1.html
Saved: AEE868_seg2.html
Saved: AEE869_seg1.html
Saved: AEE869_seg2.html
Saved: AEE873_seg1.html


<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: AFR1046_seg1.html
Saved: AFR1046_seg2.html
Saved: AFR1047_seg1.html
Saved: AFR1247_seg1.html
Saved: AFR1346_seg1.html
Saved: AFR1346_seg2.html
Saved: AFR1347_seg1.html
Saved: AUA597_seg1.html
Saved: AUA598_seg1.html
Saved: AUA599_seg1.html


<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: AUA623_seg1.html
Saved: AUA625_seg1.html
Saved: AUA628_seg1.html
Saved: AUA631_seg1.html
Saved: BAW847_seg1.html
Saved: BAW850_seg1.html


...

<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: WZZ2075_seg1.html
Saved: WZZ2075_seg2.html
Saved: WZZ2093_seg1.html
Saved: WZZ2094_seg1.html
Saved: WZZ2094_seg2.html
Saved: WZZ2094_seg3.html
Saved: WZZ2097_seg1.html
Saved: WZZ2097_seg2.html
Saved: WZZ2098_seg1.html
Saved: WZZ2467_seg1.html


<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: WZZ7941_seg1.html
Saved: WZZ7941_seg2.html
Saved: WZZ92_seg1.html

Wygenerowano 955 plików HTML w katalogu ./WYSWIETLONE_LOTY/


<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-3-24c2bf4c3269>:94: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")


In [6]:
import os
import json
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

import folium
from folium import Popup
from folium.features import DivIcon

# ------------------------------
# 1. Funkcja obliczająca odległość (haversine)
# ------------------------------
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0  # Promień Ziemi w km
    phi1, phi2 = radians(lat1), radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)
    a = sin(delta_phi / 2)**2 + cos(phi1) * cos(phi2) * sin(delta_lambda / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

# ------------------------------
# 2. Profile prędkości/Mach dla typów samolotów
# ------------------------------
aircraft_speeds = {
    "A20N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A21N": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A319": {"FL_CRUISE": 37000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A320": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A321": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "A330": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "A339": {"FL_CRUISE": 41000, "Vcl1": 310, "Mcl": 0.82, "Vcr2": 320, "Mcr": 0.82, "Vdes1": 310, "Mdes": 0.82},
    "AT72": {"FL_CRUISE": 25000, "Vcl1": 170, "Mcl": 0.45, "Vcr2": 180, "Mcr": 0.45, "Vdes1": 170, "Mdes": 0.45},
    "B38M": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B737": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B738": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "B753": {"FL_CRUISE": 39000, "Vcl1": 290, "Mcl": 0.80, "Vcr2": 300, "Mcr": 0.80, "Vdes1": 290, "Mdes": 0.80},
    "B77W": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.84, "Vcr2": 320, "Mcr": 0.84, "Vdes1": 310, "Mdes": 0.84},
    "B788": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "B789": {"FL_CRUISE": 43000, "Vcl1": 310, "Mcl": 0.85, "Vcr2": 320, "Mcr": 0.85, "Vdes1": 310, "Mdes": 0.85},
    "BCS3": {"FL_CRUISE": 41000, "Vcl1": 290, "Mcl": 0.78, "Vcr2": 300, "Mcr": 0.78, "Vdes1": 290, "Mdes": 0.78},
    "C25A": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C25B": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "C56X": {"FL_CRUISE": 45000, "Vcl1": 250, "Mcl": 0.75, "Vcr2": 260, "Mcr": 0.75, "Vdes1": 250, "Mdes": 0.75},
    "CRJ9": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "CRJX": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E170": {"FL_CRUISE": 39000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E190": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E195": {"FL_CRUISE": 41000, "Vcl1": 280, "Mcl": 0.78, "Vcr2": 290, "Mcr": 0.78, "Vdes1": 280, "Mdes": 0.78},
    "E295": {"FL_CRUISE": 31000, "Vcl1": 250, "Mcl": 0.65, "Vcr2": 260, "Mcr": 0.65, "Vdes1": 250, "Mdes": 0.65},
    "E50P": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "E75L": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "E75S": {"FL_CRUISE": 41000, "Vcl1": 270, "Mcl": 0.75, "Vcr2": 280, "Mcr": 0.75, "Vdes1": 270, "Mdes": 0.75},
    "EA50": {"FL_CRUISE": 41000, "Vcl1": 250, "Mcl": 0.70, "Vcr2": 260, "Mcr": 0.70, "Vdes1": 250, "Mdes": 0.70},
    "GLF5": {"FL_CRUISE": 51000, "Vcl1": 300, "Mcl": 0.85, "Vcr2": 310, "Mcr": 0.85, "Vdes1": 300, "Mdes": 0.85}
}


# ------------------------------
# 3. Funkcja wykrywająca fazy lotu + grupowanie bloków faz
# ------------------------------
def detect_phases(grp):
    grp = grp.reset_index(drop=True)
    grp[['alt', 'gs', 'vertRate']] = grp[['alt', 'gs', 'vertRate']].astype(float)
    grp['phase'] = pd.NA

    ac_type = grp['aircrafttype'].iloc[0]
    params = aircraft_speeds.get(ac_type)
    if not params:
        grp['phase'] = "Unknown"
    else:
        FL = params["FL_CRUISE"]
        Vcl1, Vcr2, Vdes1 = params["Vcl1"], params["Vcr2"], params["Vdes1"]

        for i, row in grp.iterrows():
            alt, gs, roc = row['alt'], row['gs'], row['vertRate']
            # climb
            if 10 <= alt <= FL - 4000 and roc >= 150:
                grp.at[i, 'phase'] = "Lower Climb" if gs <= Vcl1 else "Upper Climb"
            # cruise
            elif abs(alt - FL) <= 4000 and abs(roc) < 250:
                grp.at[i, 'phase'] = "Lower Cruise" if gs <= Vcr2 else "Upper Cruise"
            # descent
            elif 100 <= alt <= FL - 4000 and roc <= -150:
                if gs <= Vdes1 or (alt < 10000 and gs <= 250):
                    grp.at[i, 'phase'] = "Lower Descent"
                else:
                    grp.at[i, 'phase'] = "Upper Descent"
            else:
                grp.at[i, 'phase'] = pd.NA

        grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")

    # --- nowa kolumna numerująca kolejne jednolite bloki fazy ---
    grp['phase_group'] = (grp['phase'] != grp['phase'].shift()).cumsum()

    return grp

# ------------------------------
# 4. Wczytanie i wstępne czyszczenie danych
# ------------------------------
with open('all_flights_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
df = df[df['type'] == 'position'].dropna(
    subset=['lat','lon','alt','gs','vertRate','aircrafttype','clock','id','orig','dest','ident']
)
df = df.astype({'lat': float, 'lon': float, 'alt': float, 'gs': float, 'vertRate': float, 'clock': int})
df = df.sort_values(['ident','clock'])
# <-- tutaj dodajemy operator_id
df['operator_id'] = df['ident'].str[:3]

# ------------------------------
# 5. Segmentacja lotów
# ------------------------------
all_segments = []
for ident, group in df.groupby('ident', group_keys=False):
    group = group.reset_index(drop=True)
    segments = []
    current = [group.loc[0]]
    for i in range(1, len(group)):
        prev, curr = group.loc[i-1], group.loc[i]
        dist = haversine(prev.lat, prev.lon, curr.lat, curr.lon)
        dt = curr.clock - prev.clock
        if prev.alt > 30000:
            d_thresh, t_thresh = 100, 600
        elif prev.alt > 20000:
            d_thresh, t_thresh = 70, 500
        elif prev.alt > 10000:
            d_thresh, t_thresh = 50, 400
        else:
            d_thresh, t_thresh = 40, 300

        if dist > d_thresh or dt > t_thresh:
            segments.append(pd.DataFrame(current))
            current = [curr]
        else:
            current.append(curr)
    if current:
        segments.append(pd.DataFrame(current))

    for idx, seg in enumerate(segments, start=1):
        # pomijamy segmenty krótsze niż 4 rekordy
        if len(seg) < 4:
            continue
        # wykrywamy fazy i grupy faz
        seg = detect_phases(seg)
        all_segments.append((ident, idx, seg))

print(f"Znaleziono {len(all_segments)} segmentów w locie.")


# ------------------------------
# 6. Generowanie HTML z mapami (segmenty >= 4 rekordów)
# ------------------------------
out_dir = 'WYSWIETLONE_LOTY'
os.makedirs(out_dir, exist_ok=True)
phase_colors = {
    'Lower Climb':'blue','Upper Climb':'cyan',
    'Lower Cruise':'green','Upper Cruise':'darkgreen',
    'Upper Descent':'orange','Lower Descent':'red',
    'Unknown':'gray'
}

for ident, seg_idx, seg in all_segments:
    # pomijamy segmenty krótsze niż 4 rekordy
    if len(seg) < 4:
        continue

    seg = detect_phases(seg)
    start = [seg.iloc[0].lat, seg.iloc[0].lon]
    m = folium.Map(location=start, zoom_start=6)

    # rysujemy każdy jednolity blok fazy
    for _, sub in seg.groupby('phase_group'):
        pts = sub[['lat','lon']].values.tolist()
        ph = sub.iloc[0]['phase']
        clr = phase_colors.get(ph, 'gray')
        folium.PolyLine(pts, color=clr, weight=3, opacity=0.8).add_to(m)

        # markery start/end każdego bloku
        for label, row in [('start', sub.iloc[0]), ('end', sub.iloc[-1])]:
            popup = Popup(
                f"{ph} ({label})<br>"
                f"time: {row['clock']}<br>"
                f"alt: {row['alt']} ft<br>"
                f"gs: {row['gs']} kt<br>"
                f"roc: {row['vertRate']} fpm<br>"
                f"grp: {row['phase_group']}",
                max_width=220
            )
            icon = '●' if label == 'start' else '◆'
            folium.Marker(
                [row.lat, row.lon],
                popup=popup,
                icon=DivIcon(html=f"<div style='font-size:8pt;color:{clr}'>{icon}</div>")
            ).add_to(m)

    filename = f"{ident}_seg{seg_idx}.html"
    m.save(os.path.join(out_dir, filename))
    print(f"Saved: {filename}")

print(f"\nWygenerowano {len(os.listdir(out_dir))} plików HTML w katalogu ./{out_dir}/")


<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Znaleziono 880 segmentów w locie.
Saved: ABB26_seg1.html
Saved: ABB26_seg2.html
Saved: ABB26_seg3.html
Saved: ABB26_seg4.html
Saved: ABY749_seg1.html
Saved: AEE868_seg2.html


<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: AEE869_seg1.html
Saved: AEE869_seg2.html
Saved: AEE873_seg1.html
Saved: AFR1046_seg1.html
Saved: AFR1047_seg1.html
Saved: AFR1247_seg1.html
Saved: AFR1346_seg1.html


<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

...

<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: WZZ1826_seg2.html
Saved: WZZ1863_seg1.html
Saved: WZZ1864_seg1.html
Saved: WZZ1864_seg2.html
Saved: WZZ2001_seg1.html
Saved: WZZ2009_seg1.html
Saved: WZZ2009_seg2.html
Saved: WZZ2009_seg3.html
Saved: WZZ2010_seg1.html
Saved: WZZ2051_seg2.html
Saved: WZZ2052_seg1.html


<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

Saved: WZZ2063_seg1.html
Saved: WZZ2063_seg2.html
Saved: WZZ2071_seg1.html
Saved: WZZ2075_seg1.html
Saved: WZZ2075_seg2.html
Saved: WZZ2093_seg1.html
Saved: WZZ2094_seg1.html
Saved: WZZ2094_seg2.html
Saved: WZZ2094_seg3.html
Saved: WZZ2097_seg1.html
Saved: WZZ2097_seg2.html
Saved: WZZ2098_seg1.html
Saved: WZZ2467_seg1.html
Saved: WZZ7941_seg1.html
Saved: WZZ7941_seg2.html

Wygenerowano 955 plików HTML w katalogu ./WYSWIETLONE_LOTY/


<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='ffill').fillna(method='bfill').fillna("Unknown")
<ipython-input-6-e10e7dbed28a>:93: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  grp['phase'] = grp['phase'].fillna(method='f

In [17]:
# ------------------------------
# 7. Agregacja średnich prędkości per segment → nowy DataFrame
# ------------------------------
records = []
for ident, seg_idx, seg in all_segments:
    orig = seg['orig'].iloc[0]
    dest = seg['dest'].iloc[0]
    operator = seg['operator_id'].iloc[0]
    ac_type = seg['aircrafttype'].iloc[0]

    # dla każdej wyróżnionej grupy fazy
    for grp_id, sub in seg.groupby('phase_group'):
        phase = sub['phase'].iloc[0]

        # filtrujemy nieinteresujące nas starty/lądowania
        if phase in ['Lower Climb', 'Upper Climb'] and not orig.startswith('EP'):
            continue
        if phase in ['Lower Descent', 'Upper Descent'] and not dest.startswith('EP'):
            continue
        # Lower/Upper Cruise – zawsze bierzemy

        avg_speed = sub['gs'].mean()  # średnia prędkość GS
        records.append({
            'aircrafttype': ac_type,
            'orig': orig,
            'dest': dest,
            'operator_id': operator,
            'phase': phase,
            'phase_group': grp_id,
            'avg_speed': avg_speed
        })

# z listy rekordów budujemy ostateczny DataFrame
df_avg_speeds = pd.DataFrame(records)

# usuwamy rekordy, gdzie faza jest Unknown
df_avg_speeds = df_avg_speeds[df_avg_speeds['phase'] != 'Unknown']

# przykładowe podgląd pierwszych wierszy
print(df_avg_speeds.head(5))


  aircrafttype  orig  dest operator_id         phase  phase_group   avg_speed
0         A339  EPWA  KJFK         ABB   Lower Climb            1  281.666667
1         A339  EPWA  KJFK         ABB   Upper Climb            2  397.400000
5         A320  EPKK  OMSJ         ABY  Upper Cruise            1  476.272727
6         A320  LGAV  EPKK         AEE  Upper Cruise            1  459.500000
8         A321  EPWA  LGAV         AEE  Upper Cruise            1  450.312500


In [24]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Zakładamy, że df_avg_speeds to DataFrame z danymi wejściowymi

# Filtrujemy dane, aby uwzględniały tylko loty związane z wybranymi lotniskami
warsaw_airports = ['EPWA', 'EPMO', 'EPRA']
df_filtered = df_avg_speeds[
    df_avg_speeds['orig'].isin(warsaw_airports) | df_avg_speeds['dest'].isin(warsaw_airports)
]

# Wyświetlamy liczbę rekordów po filtracji
print(f"Liczba rekordów po filtracji: {len(df_filtered)}")

# Tworzymy tabelę przestawną
pivot = df_filtered.pivot_table(
    index=['operator_id', 'aircrafttype', 'orig', 'dest', 'phase'],
    values='avg_speed',
    aggfunc='mean'
)

# Funkcja pomocnicza do obliczania średniej prędkości dla danego wzorca
def mean_for_pattern(op, ac, adep, phase, pattern):
    try:
        sub = pivot.loc[
            (slice(None) if op == '***' else op,
             ac,
             adep,
             slice(None),
             phase)
        ]
    except KeyError:
        return np.nan

    if isinstance(sub, pd.Series):
        sub = sub.to_frame().T

    if pattern != '****':
        prefix = pattern.rstrip('*')
        mask = sub.index.get_level_values('dest').str.startswith(prefix)
        sub = sub[mask]

    return sub['avg_speed'].mean()

# Przykładowe dane Mach dla różnych typów samolotów
mach_cruise = {'A320': 0.78, 'A321': 0.80, 'DH8D': 0.73}
mach_climb = {'A320': 0.75, 'A321': 0.76, 'DH8D': 0.72}
mach_descent = {'A320': 0.77, 'A321': 0.78, 'DH8D': 0.74}

# Generowanie wierszy z paskiem postępu
rows = []
operators = list(df_filtered['operator_id'].dropna().unique()) + ['***']
adeps = sorted(df_filtered['orig'][df_filtered['orig'].str.startswith('EP')].unique())
ac_types = df_filtered['aircrafttype'].dropna().unique()
patterns = sorted(set(df_filtered['dest']) |
                  {d[0] + '***' for d in df_filtered['dest']} |
                  {d[:2] + '**' for d in df_filtered['dest']} |
                  {'****'})

total_iterations = len(operators) * len(ac_types) * len(adeps) * len(patterns)

with tqdm(total=total_iterations, desc="Generowanie tabeli") as pbar:
    for op in operators:
        for ac in ac_types:
            for adep in adeps:
                for pat in patterns:
                    uc = mean_for_pattern(op, ac, adep, 'Upper Cruise', pat)
                    lc = mean_for_pattern(op, ac, adep, 'Lower Cruise', pat)
                    ud = mean_for_pattern(op, ac, adep, 'Upper Descent', pat)
                    ld = mean_for_pattern(op, ac, adep, 'Lower Descent', pat)
                    cl = mean_for_pattern(op, ac, adep, 'Upper Climb', pat)
                    cll = mean_for_pattern(op, ac, adep, 'Lower Climb', pat)

                    if all(pd.isna(v) for v in [uc, lc, ud, ld, cl, cll]):
                        pbar.update(1)
                        continue

                    rows.append({
                        'Aircraft Operator Id.': op,
                        'Aircraft Pattern Id.': ac,
                        'ADEP': adep,
                        'ADES': pat,
                        'Upper Cruise Speed': uc,
                        'Lower Cruise Speed': lc,
                        'Cruise Mach Speed': mach_cruise.get(ac, np.nan),
                        'Upper Climb Speed': cl,
                        'Lower Climb Speed': cll,
                        'Climb Mach Speed': mach_climb.get(ac, np.nan),
                        'Upper Descent Speed': ud,
                        'Lower Descent Speed': ld,
                        'Descent Mach Speed': mach_descent.get(ac, np.nan),
                    })
                    pbar.update(1)

# Tworzymy DataFrame z wynikami
df_summary = pd.DataFrame(rows).sort_values(
    ['Aircraft Operator Id.', 'Aircraft Pattern Id.', 'ADEP', 'ADES']
).reset_index(drop=True)

# Wyświetlamy wynikową tabelę
print(df_summary)


Liczba rekordów po filtracji: 407


Generowanie tabeli: 100%|██████████| 742896/742896 [54:02<00:00, 229.10it/s]

    Aircraft Operator Id. Aircraft Pattern Id.  ADEP  ADES  \
0                     ***                 A20N  EPWA  ****   
1                     ***                 A20N  EPWA  L***   
2                     ***                 A20N  EPWA  LG**   
3                     ***                 A20N  EPWA  LGAV   
4                     ***                 A21N  EPWA  ****   
..                    ...                  ...   ...   ...   
662                   WZZ                 A321  EPWA  LGKR   
663                   WZZ                 A321  EPWA  LI**   
664                   WZZ                 A321  EPWA  LIBD   
665                   WZZ                 A321  EPWA  LICC   
666                   WZZ                 A321  EPWA  LIME   

     Upper Cruise Speed  Lower Cruise Speed  Cruise Mach Speed  \
0            456.000000                 NaN                NaN   
1            456.000000                 NaN                NaN   
2            456.000000                 NaN              

In [25]:
df_summary.to_csv('df_summary.csv', index=False)


In [27]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Lista kodów lotnisk warszawskich
warsaw_airports = ['EPWA', 'EPMO', 'EPRA']

# Filtrowanie tylko na przyloty do Warszawy
df_filtered = df_avg_speeds[df_avg_speeds['dest'].isin(warsaw_airports)]

# Liczba rekordów po filtracji
print(f"Liczba rekordów po filtracji: {len(df_filtered)}")

# Tworzenie tabeli przestawnej
pivot = df_filtered.pivot_table(
    index=['operator_id', 'aircrafttype', 'orig', 'dest', 'phase'],
    values='avg_speed',
    aggfunc='mean'
)

# Funkcja pomocnicza do obliczeń
def mean_for_pattern(op, ac, ades, phase, pattern):
    try:
        sub = pivot.loc[
            (slice(None) if op == '***' else op,
             ac,
             slice(None),
             ades,
             phase)
        ]
    except KeyError:
        return np.nan

    if isinstance(sub, pd.Series):
        sub = sub.to_frame().T

    if pattern != '****':
        prefix = pattern.rstrip('*')
        mask = sub.index.get_level_values('orig').str.startswith(prefix)
        sub = sub[mask]

    return sub['avg_speed'].mean()

# Przykładowe dane Mach
mach_cruise = {'A320': 0.78, 'A321': 0.80, 'DH8D': 0.73}
mach_climb = {'A320': 0.75, 'A321': 0.76, 'DH8D': 0.72}
mach_descent = {'A320': 0.77, 'A321': 0.78, 'DH8D': 0.74}

# Przygotowanie do iteracji
rows = []
operators = list(df_filtered['operator_id'].dropna().unique()) + ['***']
ades_list = sorted(df_filtered['dest'].unique())  # docelowe lotniska – tylko warszawskie
ac_types = df_filtered['aircrafttype'].dropna().unique()
patterns = sorted(set(df_filtered['orig']) |
                  {d[0] + '***' for d in df_filtered['orig']} |
                  {d[:2] + '**' for d in df_filtered['orig']} |
                  {'****'})

total_iterations = len(operators) * len(ac_types) * len(ades_list) * len(patterns)

with tqdm(total=total_iterations, desc="Generowanie tabeli") as pbar:
    for op in operators:
        for ac in ac_types:
            for ades in ades_list:
                for pat in patterns:
                    uc = mean_for_pattern(op, ac, ades, 'Upper Cruise', pat)
                    lc = mean_for_pattern(op, ac, ades, 'Lower Cruise', pat)
                    ud = mean_for_pattern(op, ac, ades, 'Upper Descent', pat)
                    ld = mean_for_pattern(op, ac, ades, 'Lower Descent', pat)
                    cl = mean_for_pattern(op, ac, ades, 'Upper Climb', pat)
                    cll = mean_for_pattern(op, ac, ades, 'Lower Climb', pat)

                    if all(pd.isna(v) for v in [uc, lc, ud, ld, cl, cll]):
                        pbar.update(1)
                        continue

                    rows.append({
                        'Aircraft Operator Id.': op,
                        'Aircraft Pattern Id.': ac,
                        'ADEP': pat,
                        'ADES': ades,
                        'Upper Cruise Speed': uc,
                        'Lower Cruise Speed': lc,
                        'Cruise Mach Speed': mach_cruise.get(ac, np.nan),
                        'Upper Climb Speed': cl,
                        'Lower Climb Speed': cll,
                        'Climb Mach Speed': mach_climb.get(ac, np.nan),
                        'Upper Descent Speed': ud,
                        'Lower Descent Speed': ld,
                        'Descent Mach Speed': mach_descent.get(ac, np.nan),
                    })
                    pbar.update(1)

# Tworzenie końcowego DataFrame
df_summary = pd.DataFrame(rows).sort_values(
    ['Aircraft Operator Id.', 'Aircraft Pattern Id.', 'ADEP', 'ADES']
).reset_index(drop=True)

# Wyświetlenie wyników
print(df_summary)
df_summary.to_csv('df_summary.csv', index=False)



Liczba rekordów po filtracji: 211


Generowanie tabeli: 100%|██████████| 92752/92752 [06:41<00:00, 230.82it/s]

    Aircraft Operator Id. Aircraft Pattern Id.  ADEP  ADES  \
0                     ***                 A21N  ****  EPWA   
1                     ***                 A21N  L***  EPWA   
2                     ***                 A21N  LH**  EPWA   
3                     ***                 A21N  LHBP  EPWA   
4                     ***                 A21N  LI**  EPWA   
..                    ...                  ...   ...   ...   
507                   WZZ                 A321  LICC  EPWA   
508                   WZZ                 A321  LL**  EPWA   
509                   WZZ                 A321  LLBG  EPWA   
510                   WZZ                 A321  LP**  EPWA   
511                   WZZ                 A321  LPPT  EPWA   

     Upper Cruise Speed  Lower Cruise Speed  Cruise Mach Speed  \
0            471.166667                 NaN                NaN   
1                   NaN                 NaN                NaN   
2                   NaN                 NaN              